<a href="https://colab.research.google.com/github/Ashi69-eng/TriVitaX/blob/main/MEDTECH_USING_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd # Import the pandas library
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

display(df.head())

Saving Dengue-Dataset.csv to Dengue-Dataset.csv


,Gender,Age,Hemoglobin(g/dl),Neutrophils(%),Lymphocytes(%),Monocytes(%),Eosinophils(%),RBC,HCT(%),MCV(fl),MCH(pg),MCHC(g/dl),RDW-CV(%),Total Platelet Count(/cumm),MPV(fl),PDW(%),PCT(%),Total WBC count(/cumm),Result
0,Male,21,14.8,48,47,3,2,5,48.00,96.0,29.60,30.8,11.6,112000,10.70,15.40,0.120,5100,positive
1,Male,30,15.0,47,49,6,3,5,49.80,96.1,28.40,29.5,11.8,96000,10.60,15.80,0.121,4500,positive
2,Male,51,16.3,41,48,4,5,5,50.10,93.5,31.30,32.7,13.5,184000,10.40,16.40,0.130,6000,negative
3,Female,26,12.3,46,49,7,5,5,44.00,90.0,30.50,30.5,14.7,167000,8.10,17.10,0.110,5000,negative
4,Male,35,16.1,45,46,4,4,5,50.53,91.0,29.12,29.2,15.2,155000,10.52,12.34,0.150,4600,negative


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("Dengue-Dataset.csv")

# Display columns for verification
print("\nColumns in dataset:", df.columns.tolist())

# ---- Step 1: Define Hemoglobin threshold ----
HEMOGLOBIN_THRESHOLD = 12.0

# Check if hemoglobin column exists
possible_hemo_cols = [col for col in df.columns if 'hemo' in col.lower()]
if not possible_hemo_cols:
    raise ValueError("❌ Hemoglobin column not found. Please check your dataset column names.")
hemoglobin_col = possible_hemo_cols[0]
print(f" Using Hemoglobin column: {hemoglobin_col}")

# ---- Step 2: Create binary target column (Low Hemoglobin) ----
df['Low_Hemoglobin'] = (df[hemoglobin_col] < HEMOGLOBIN_THRESHOLD).astype(int)

# ---- Step 3: Define features (X) and target (Y) ----
# Try to drop only if they exist in the dataset
drop_cols = [col for col in ['Result_Encoded', hemoglobin_col, 'Low_Hemoglobin'] if col in df.columns]
X_hemoglobin = df.drop(columns=drop_cols)
Y_hemoglobin = df['Low_Hemoglobin']

# ---- Step 4: Handle non-numeric columns ----
X_hemoglobin = pd.get_dummies(X_hemoglobin, drop_first=True)

# ---- Step 5: Handle missing values ----
X_hemoglobin = X_hemoglobin.fillna(0)

# ---- Step 6: Split dataset ----
X_train, X_test, Y_train, Y_test = train_test_split(
    X_hemoglobin, Y_hemoglobin, test_size=0.3, random_state=42, stratify=Y_hemoglobin
)

# ---- Step 7: Feature scaling ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---- Step 8: Train SVM Model ----
model_hemoglobin = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
model_hemoglobin.fit(X_train_scaled, Y_train)

# ---- Step 9: Evaluate model ----
Y_pred = model_hemoglobin.predict(X_test_scaled)
accuracy = accuracy_score(Y_test, Y_pred)

# ---- Step 10: Output results ----
print("\n--- Low Hemoglobin Prediction Model (Red Port) ---")
print(f"Prediction Target: Low Hemoglobin (< {HEMOGLOBIN_THRESHOLD} g/dl)")
print(f"Model: Support Vector Machine (SVM)")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))



Columns in dataset: ['Gender', 'Age', 'Hemoglobin(g/dl)', 'Neutrophils(%)', 'Lymphocytes(%)', 'Monocytes(%)', 'Eosinophils(%)', 'RBC', 'HCT(%)', 'MCV(fl)', 'MCH(pg)', 'MCHC(g/dl)', 'RDW-CV(%)', 'Total Platelet Count(/cumm)', 'MPV(fl)', 'PDW(%)', 'PCT(%)', 'Total WBC count(/cumm)', 'Result']
 Using Hemoglobin column: Hemoglobin(g/dl)

--- Low Hemoglobin Prediction Model (Red Port) ---
Prediction Target: Low Hemoglobin (< 12.0 g/dl)
Model: Support Vector Machine (SVM)
Accuracy: 93.22%

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       424
           1       1.00      0.06      0.11        33

    accuracy                           0.93       457
   macro avg       0.97      0.53      0.54       457
weighted avg       0.94      0.93      0.90       457



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("Dengue-Dataset.csv")

# Display columns for verification
print("\nColumns in dataset:", df.columns.tolist())

# ---- Step 1: Define platelet threshold ----
PLATELET_THRESHOLD = 150000

# Try to detect platelet count column automatically
possible_platelet_cols = [col for col in df.columns if 'platelet' in col.lower()]
if not possible_platelet_cols:
    raise ValueError("❌ Platelet count column not found. Please check dataset column names.")
platelet_col = possible_platelet_cols[0]
print(f"✅ Using Platelet column: {platelet_col}")

# ---- Step 2: Create binary target (Low Platelet) ----
df['Low_Platelets'] = (df[platelet_col] < PLATELET_THRESHOLD).astype(int)

# ---- Step 3: Define features (X) and target (Y) ----
drop_cols = [col for col in ['Result_Encoded', platelet_col, 'Low_Platelets'] if col in df.columns]
X_platelets = df.drop(columns=drop_cols)
Y_platelets = df['Low_Platelets']

# ---- Step 4: Handle non-numeric columns ----
X_platelets = pd.get_dummies(X_platelets, drop_first=True)

# ---- Step 5: Handle missing values ----
X_platelets = X_platelets.fillna(0)

# ---- Step 6: Split data ----
X_train, X_test, Y_train, Y_test = train_test_split(
    X_platelets, Y_platelets, test_size=0.3, random_state=42, stratify=Y_platelets
)

# ---- Step 7: Scale features ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---- Step 8: Train SVM model ----
model_platelets = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
model_platelets.fit(X_train_scaled, Y_train)

# ---- Step 9: Predict ----
Y_pred = model_platelets.predict(X_test_scaled)

# ---- Step 10: Evaluate performance ----
accuracy = accuracy_score(Y_test, Y_pred)

print("\n--- Low Platelet Prediction Model (Yellow Port) ---")
print(f"Prediction Target: Thrombocytopenia (< {PLATELET_THRESHOLD} /cumm)")
print(f"Model: Support Vector Machine (SVM)")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))



Columns in dataset: ['Gender', 'Age', 'Hemoglobin(g/dl)', 'Neutrophils(%)', 'Lymphocytes(%)', 'Monocytes(%)', 'Eosinophils(%)', 'RBC', 'HCT(%)', 'MCV(fl)', 'MCH(pg)', 'MCHC(g/dl)', 'RDW-CV(%)', 'Total Platelet Count(/cumm)', 'MPV(fl)', 'PDW(%)', 'PCT(%)', 'Total WBC count(/cumm)', 'Result']
✅ Using Platelet column: Total Platelet Count(/cumm)

--- Low Platelet Prediction Model (Yellow Port) ---
Prediction Target: Thrombocytopenia (< 150000 /cumm)
Model: Support Vector Machine (SVM)
Accuracy: 73.30%

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.95      0.80       263
           1       0.87      0.44      0.58       194

    accuracy                           0.73       457
   macro avg       0.78      0.69      0.69       457
weighted avg       0.77      0.73      0.71       457

